# TinyTTS: train the 16 kHz LJSpeech voice on a Kaggle GPU

Trains or fine-tunes the `ljspeech16k` model of [TinyTTS](https://github.com/pschatzmann/TinyTTS)
(16 kHz output, LJSpeech voice) on a Kaggle GPU. At the end it exports a `weights.bin` that the
C++ library loads directly (`tts.setWeights(file)`, `tinytts --weights ...`).

**Before running**, in the notebook's sidebar under *Settings*:
- **Accelerator:** *GPU T4 x2* or *GPU P100*. The training script uses one GPU.
- **Internet:** *On*. Your Kaggle account must be phone-verified for this. The notebook clones
  GitHub and downloads LJSpeech, the released checkpoint and the tokenizer.

**How to run:** use *Save Version → Save & Run All (Commit)*. That runs in the background for up
to 12 hours, even with the browser closed, and keeps everything written to `/kaggle/working` as
the version's output. `MAX_HOURS` below stops training in time to export the result.
*Run All* in the interactive editor works too, but stops when the session times out.

**Continuing in another session:** open the notebook again and click *Add Input*. Add this
notebook's own previous output (Notebook Output Files), then run it again. It finds the
previous checkpoints and prepared data automatically (`PREVIOUS_OUTPUT = "auto"`) and resumes
where it stopped.

## 1. Settings

In [ ]:
import os

REPO_URL = "https://github.com/pschatzmann/TinyTTS"
REPO_BRANCH = "main"
# Upstream model code (tronghieuit/tiny-tts), same pinned commit as the Docker image.
UPSTREAM_URL = "https://github.com/tronghieuit/tiny-tts.git"
UPSTREAM_COMMIT = "77c278f7a4659181bd7bcf5c5e07c768ae765147"

RUN_NAME = "ljspeech16k"
SAMPLE_RATE = 16000

# Where to start a NEW run (ignored when resuming):
#   None -> the released 44.1 kHz model (HuggingFace backtracking/tiny-tts G.pth), adapted to 16 kHz
#   path -> a G_*.pth you uploaded as a Kaggle dataset, e.g. the laptop run's G_40000.pth:
#           "/kaggle/input/tinytts-ljspeech16k/G_40000.pth"
INIT_CHECKPOINT = None

# Continue a previous session: "auto" searches /kaggle/input for runs/<RUN_NAME>/G_*.pth
# (add the previous notebook version's output as input); None always starts fresh.
PREVIOUS_OUTPUT = "auto"

MAX_STEPS = 100_000   # total steps across all sessions
MAX_HOURS = 11.0      # per session: stop training here, leaving time to export (Kaggle limit: 12 h)
BATCH_SIZE = 16       # T4 / P100 (16 GB); lower it if you get CUDA out-of-memory errors
DISC_WIDTH = 1.0      # full-size HiFi-GAN discriminator, affordable on a GPU
WORKERS = 4           # DataLoader processes (Kaggle GPU sessions have 4 CPU cores)
ALLOW_CPU = False     # True: run even without a GPU (only sensible for a quick test)

# Publish the result as a new version of a Kaggle Model (kaggle.com/models), see step 7.
PUBLISH_MODEL = True
MODEL_HANDLE = "pschatzmann/tinytts/other/ljspeech16k"   # <your Kaggle user>/<model>/<framework>/<variation>
MODEL_LICENSE = "Apache 2.0"                              # same license as TinyTTS

# Kaggle paths (overridable via environment variables for local testing).
WORK = os.environ.get("TINYTTS_WORK", "/kaggle/working")
INPUT = os.environ.get("TINYTTS_INPUT", "/kaggle/input")
REPO = os.environ.get("TINYTTS_REPO_DIR", "/tmp/TinyTTS")   # code lives outside /kaggle/working (not part of the output)
TRAINING = f"{REPO}/research/training"
DATA_DIR = f"{WORK}/data/{RUN_NAME}"
RUN_DIR = f"{WORK}/runs/{RUN_NAME}"

## 2. GPU check

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  cuda:{i} {p.name}, {p.total_memory / 2**30:.1f} GB")
elif ALLOW_CPU:
    print("WARNING: no GPU -- running on CPU (ALLOW_CPU = True); expect several seconds per step")
else:
    raise SystemExit(
        "No GPU in this session. In the notebook editor open Settings -> Accelerator, choose "
        "'GPU T4 x2' or 'GPU P100', then Save Version again (the accelerator is a per-notebook "
        "setting, not taken from the imported file). Set ALLOW_CPU = True to run on CPU anyway.")

## 3. Code and dependencies

In [ ]:
import glob
import shutil
import subprocess
import sys


def run(cmd, cwd=None, log=None, filter_fn=None):
    """Runs a command, streaming its output (optionally filtered and/or teed to a log file)."""
    print("$", " ".join(cmd), flush=True)
    logf = open(log, "a") if log else None
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if logf:
            logf.write(line)
            logf.flush()
        if filter_fn is None or filter_fn(line):
            print(line, end="", flush=True)
    if logf:
        logf.close()
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {' '.join(cmd)}")


if not os.path.isdir(f"{REPO}/.git") and not os.environ.get("TINYTTS_REPO_DIR"):
    run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO])
# This notebook needs train.py --max-hours and configs/en_ljspeech16k_gpu.json.
if "max_hours" not in open(f"{TRAINING}/train.py").read() or not os.path.exists(f"{TRAINING}/configs/en_ljspeech16k_gpu.json"):
    raise SystemExit(f"{REPO_URL} ({REPO_BRANCH}) is older than this notebook -- push the current "
                     "research/training/ changes (train.py --max-hours, configs/en_ljspeech16k_gpu.json) first.")

upstream = f"{REPO}/research/tiny-tts"
if not os.path.isdir(f"{upstream}/tiny_tts"):
    # Sparse + blobless: only tiny_tts/, not the large checkpoints/ONNX files in that repo.
    os.makedirs(upstream, exist_ok=True)
    for cmd in (["git", "init", "-q"], ["git", "remote", "add", "origin", UPSTREAM_URL],
                ["git", "sparse-checkout", "set", "tiny_tts"],
                ["git", "fetch", "-q", "--depth", "1", "--filter=blob:none", "origin", UPSTREAM_COMMIT],
                ["git", "checkout", "-q", "FETCH_HEAD"]):
        run(cmd, cwd=upstream)

# Kaggle already ships a CUDA build of torch/torchaudio -- install everything else, never torch.
reqs = [l.strip() for l in open(f"{TRAINING}/requirements.txt")
        if l.strip() and not l.startswith("#") and not l.startswith(("torch", "torchaudio"))]
run([sys.executable, "-m", "pip", "install", "-q", *reqs])
# Load the English frontend once: fetches its NLTK data and the tokenizer here, in one process.
run([sys.executable, "-c", "from frontends import get_frontend; print(get_frontend('EN').text_to_ids('Hello world.')[0])"],
    cwd=TRAINING, filter_fn=lambda l: "Warning" not in l and "warnings.warn" not in l)
print("ok")

## 4. Dataset

Uses LJSpeech from an attached Kaggle dataset if there is one (any input containing
`metadata.csv` + `wavs/`). Otherwise it is downloaded (2.6 GB) to `/tmp`, so the raw dataset
doesn't end up in the notebook output. Preparing it (resampling to 16 kHz, phonemizing) takes
roughly 20–30 minutes. The prepared copy is kept in the output, so a resumed session skips this.

In [ ]:
def find_previous(*parts):
    """A path below a previous session's output in /kaggle/input, or None."""
    if PREVIOUS_OUTPUT is None:
        return None
    roots = [PREVIOUS_OUTPUT] if PREVIOUS_OUTPUT != "auto" else sorted(glob.glob(f"{INPUT}/**/runs", recursive=True))
    for root in roots:
        base = os.path.dirname(root) if PREVIOUS_OUTPUT == "auto" else root
        candidate = os.path.join(base, *parts)
        if os.path.exists(candidate):
            return candidate
    return None


if not os.path.exists(f"{DATA_DIR}/train.txt"):
    prev_data = find_previous("data", RUN_NAME)
    if prev_data and os.path.exists(f"{prev_data}/train.txt"):
        # Same absolute location as in the previous session, so the filelists' paths stay valid.
        print("copying prepared data from", prev_data)
        shutil.copytree(prev_data, DATA_DIR)
    else:
        found = [os.path.dirname(m) for m in glob.glob(f"{INPUT}/**/metadata.csv", recursive=True)
                 if os.path.isdir(os.path.join(os.path.dirname(m), "wavs"))]
        if found:
            ljspeech = found[0]
            print("using attached dataset", ljspeech)
        else:
            ljspeech = "/tmp/LJSpeech-1.1"
            if not os.path.exists(f"{ljspeech}/metadata.csv"):
                run(["bash", "-c", "curl -sSL https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2 | tar -xj -C /tmp"])
        run([sys.executable, "prepare_dataset.py", "--input", ljspeech, "--out", DATA_DIR,
             "--sample-rate", str(SAMPLE_RATE), "--workers", str(os.cpu_count())],
            cwd=TRAINING, filter_fn=lambda l: "Warning" not in l and "warn" not in l)
print(sum(1 for _ in open(f"{DATA_DIR}/train.txt")), "training utterances in", DATA_DIR)

## 5. Train

In [ ]:
if not glob.glob(f"{RUN_DIR}/G_*.pth"):
    prev_run = find_previous("runs", RUN_NAME)
    if prev_run and glob.glob(f"{prev_run}/G_*.pth"):
        print("continuing previous session from", prev_run)
        shutil.copytree(prev_run, RUN_DIR, dirs_exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
resume = bool(glob.glob(f"{RUN_DIR}/G_*.pth"))

cmd = [sys.executable, "train.py", "--config", "configs/en_ljspeech16k_gpu.json",
       "--train", f"{DATA_DIR}/train.txt", "--val", f"{DATA_DIR}/val.txt", "--run-dir", RUN_DIR,
       "--max-steps", str(MAX_STEPS), "--max-hours", str(MAX_HOURS),
       "--batch-size", str(BATCH_SIZE), "--disc-width", str(DISC_WIDTH), "--workers", str(WORKERS)]
if resume:
    cmd.append("--resume")
elif INIT_CHECKPOINT:
    cmd += ["--init-checkpoint", INIT_CHECKPOINT]
# Full log in runs/<RUN_NAME>/train.log; the notebook shows everything except library warnings.
run(cmd, cwd=TRAINING, log=f"{RUN_DIR}/train.log",
    filter_fn=lambda l: "Warning" not in l and "warnings.warn" not in l and "WeightNorm.apply" not in l)

## 6. Export for the C++ library

Exports the newest checkpoint to `weights.bin` (C++ format, 16 kHz metadata included), copied to
`/kaggle/working/ljspeech16k_weights.bin`, and synthesizes a test sentence.

In [ ]:
import re

from IPython.display import Audio, display

steps = sorted(int(re.search(r"G_(\d+)\.pth$", p).group(1)) for p in glob.glob(f"{RUN_DIR}/G_*.pth"))
latest = f"{RUN_DIR}/G_{steps[-1]}.pth"
print("exporting", latest)
export_dir = f"{RUN_DIR}/export"
run([sys.executable, "export_weights_and_vectors.py", "--checkpoint", latest, "--out-dir", export_dir],
    cwd=f"{REPO}/research", filter_fn=lambda l: "Warning" not in l and "WeightNorm" not in l)
shutil.copy(f"{export_dir}/weights.bin", f"{WORK}/{RUN_NAME}_weights.bin")
print(f"-> {WORK}/{RUN_NAME}_weights.bin (step {steps[-1]})")

sample = f"{WORK}/{RUN_NAME}_step{steps[-1]}.wav"
run([sys.executable, "synthesize.py", "--checkpoint", latest, "-o", sample,
     "--text", "Hello world! This is the sixteen kilohertz voice, trained on a Kaggle GPU."],
    cwd=TRAINING, filter_fn=lambda l: "Warning" not in l and "WeightNorm" not in l)
display(Audio(sample))

## 7. Publish as a Kaggle Model

Uploads the result as a new version of the Kaggle Model `MODEL_HANDLE`. If the model doesn't
exist yet, it is created as a **private** model; you can make it public later on its page. A
version contains:
- `ljspeech16k_weights.bin`, for the C++ library;
- `G_<step>.pth`, the training checkpoint, to continue training from (`INIT_CHECKPOINT`);
- a test WAV file and a `README.md`.

Uploading needs your Kaggle API token as a **notebook secret**. In the editor, open *Add-ons →
Secrets*, add a secret named `KAGGLE_API_TOKEN` with the token (*kaggle.com → Settings → API →
Create New Token*), and tick it for this notebook. Without it, or if the upload fails, this
step only prints what's missing; the files are still in the output.

In [ ]:
import datetime

model_dir = f"{WORK}/model"
shutil.rmtree(model_dir, ignore_errors=True)
os.makedirs(model_dir)
shutil.copy(f"{WORK}/{RUN_NAME}_weights.bin", model_dir)
shutil.copy(latest, model_dir)
shutil.copy(sample, model_dir)
readme = f"""# TinyTTS {RUN_NAME}

Voice model for [TinyTTS]({REPO_URL}) (C++ text-to-speech for microcontrollers):
{SAMPLE_RATE} Hz output, LJSpeech voice, training step {steps[-1]}
({datetime.date.today().isoformat()}).

| File | Use |
|---|---|
| `{RUN_NAME}_weights.bin` | Model weights for the C++ library: `tts.setWeights(file)`, or `tinytts --weights {RUN_NAME}_weights.bin` |
| `{os.path.basename(latest)}` | Training checkpoint (PyTorch): continue training with `train.py --init-checkpoint` |
| `{os.path.basename(sample)}` | Sample output |

The dictionary and G2P data are not included: they're the same English files for every voice
(`data/` in the TinyTTS repository). See `docs/training.md` there for how this model was trained.
"""
with open(f"{model_dir}/README.md", "w") as f:
    f.write(readme)
print("model files:", sorted(os.listdir(model_dir)))

if PUBLISH_MODEL:
    try:
        from kaggle_secrets import UserSecretsClient  # only exists on Kaggle
        os.environ.setdefault("KAGGLE_API_TOKEN", UserSecretsClient().get_secret("KAGGLE_API_TOKEN"))
    except Exception as e:
        print("no KAGGLE_API_TOKEN notebook secret:", type(e).__name__, "-- trying without it")
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kagglehub"])
        import kagglehub

        kagglehub.model_upload(MODEL_HANDLE, model_dir, license_name=MODEL_LICENSE,
                               version_notes=f"step {steps[-1]}, {SAMPLE_RATE} Hz")
        print(f"published: https://www.kaggle.com/models/{MODEL_HANDLE.rsplit('/', 2)[0]}")
    except Exception as e:
        print(f"model upload failed ({type(e).__name__}: {e})")
        print("Add the KAGGLE_API_TOKEN secret (see above) and check MODEL_HANDLE's owner; "
              f"the files are in {model_dir} either way.")

## 8. Using the result

Download `ljspeech16k_weights.bin` from the saved version's **Output** tab, or from the
published model: on the model's page, or from a shell with the `kaggle` CLI:

```bash
kaggle models instances versions download pschatzmann/tinytts/other/ljspeech16k/<version> -p ./model --untar
```

Then, in your TinyTTS checkout:

```bash
cp ljspeech16k_weights.bin data/ljspeech16k_weights.bin            # replace the bundled 16 kHz model
./build/desktop/tinytts --weights data/ljspeech16k_weights.bin "Hello world!"
cd research && python export_headers.py --weights ../data/ljspeech16k_weights.bin --name ljspeech16k   # refresh the C++ header
```

`runs/ljspeech16k/samples/step_<N>.wav` in the output has a sample every 1,000 steps;
`runs/ljspeech16k/train.log` is the full training log. To train further, see *Continuing in
another session* at the top.